# Virtual Topologies and Neighborhood Collectives

In a domain decomposition, every process works on one tile of a multi-dimensional domain and communicates mostly with the processes that own the neighboring tiles.
This notebook introduces the MPI features for such patterns:

* **virtual topologies** arrange the processes in a grid, so that MPI computes coordinates and neighbors for you,
* **neighborhood collectives** communicate with all neighbors of a topology in one call.

The ghost cell exchange in this notebook sends the rows and columns of a 2D grid with the derived types `MPI_Type_contiguous()` and `MPI_Type_vector()`, and its debugging helper uses `MPI_Type_create_subarray()` — all introduced in [Derived Data Types](datatypes.ipynb).

**After working through this notebook you will be able to**

* create Cartesian process grids with `MPI_Dims_create()` and `MPI_Cart_create()`, and find coordinates and neighbors,
* split a grid into row and column communicators with `MPI_Cart_sub()`,
* exchange data with all neighbors of a grid with neighborhood collectives such as `MPI_Neighbor_allgather()`,
* implement a 2D ghost cell exchange with derived types — with point-to-point calls and with `MPI_Neighbor_alltoallw()`.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The C code cells are compiled and run with the `%%mpi` cell magic: a cell is the **body of `main()`**, with `rank` and `size` already set up, unless you pass `-r`.
The options used in this notebook are

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4); some cells need exactly 12 |
| `-t` | add timing code (printed by rank 0) |
| `-i HEADER …` | include additional headers |
| `-f FLAG …` | pass additional flags to the compiler, without the leading `-` (e.g. `-f Icode` for the include path `-Icode`) |
| `-o FILE` | where to write the source code |

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

## Virtual Topologies

Many problems are multi-dimensional: a 2D domain of 4000 × 3000 cells is split into 12 tiles of 1000 × 1000 cells, one per process.
Which rank gets which tile? Who is whose neighbor? Which rank is at a boundary?

A **virtual topology** gives the processes of a communicator a naming scheme that fits the communication pattern:

* MPI maps the ranks to coordinates in a process grid — you map the parts of your arrays to the coordinates,
* the code becomes simpler: coordinates and neighbors are computed by MPI,
* MPI may use the topology to place the processes well on the hardware.

### Cartesian Topologies

```c
int MPI_Cart_create(MPI_Comm comm_old, int ndims, const int dims[], const int periods[],
                    int reorder, MPI_Comm *comm_cart);
```

| Argument | Meaning |
|---|---|
| `ndims` | number of dimensions of the process grid |
| `dims[i]` | number of processes in dimension `i` |
| `periods[i]` | true if dimension `i` is periodic (wraps around) |
| `reorder` | true if MPI may assign different ranks in `comm_cart` than in `comm_old` |
| `comm_cart` | the new communicator with the Cartesian topology |

![A 4 by 3 process grid: dimension 0 has 4 processes and is periodic, dimension 1 has 3. Rank r has the coordinates (r / 3, r mod 3); e.g., rank 3 is at (1, 0) and rank 5 at (1, 2)](img/cart-grid.svg)

* The ranks are assigned to the coordinates with the **last dimension varying fastest**, like the elements of a C array `int grid[4][3]`: rank = c0 · dims[1] + c1 in 2D.
  The figure draws dimension 0 horizontally, like x, and dimension 1 vertically, like y — as in the ghost cell exchange below, where dimension 0 runs along `i`.
  Consecutive ranks therefore run *up the columns* of the figure (0, 1, 2 in the first column), although the numbering is row-major in terms of the coordinates (c0, c1).
* `MPI_Cart_create()` is collective. If `comm_old` has more processes than the grid, the remaining ones get `MPI_COMM_NULL`; fewer processes are an error.
* With `reorder = 1`, MPI may renumber the processes to fit the hardware — always use the rank in `comm_cart` afterwards, not the one in `MPI_COMM_WORLD`.

### Choosing the Grid: `MPI_Dims_create()`

```c
int MPI_Dims_create(int nnodes, int ndims, int dims[]);
```

computes a balanced distribution of `nnodes` processes to `ndims` dimensions.
Only entries of `dims` that are **0** are set; positive entries are kept as given — so `dims` must be initialized before the call.
The product of the positive entries must divide `nnodes`; otherwise, the call is erroneous.

> 🤔 **Predict:** Which grids does `MPI_Dims_create()` choose for 7 processes in 2D? For 12 processes with `dims[1] = 2` fixed?

In [ ]:
%%mpi -n 1 -o code/topo-dims-create.c
int nprocs[] = { 4, 6, 7, 12, 16, 24 };

for (int k = 0; k < 6; ++k) {
    int dims2[2] = { 0, 0 }, dims3[3] = { 0, 0, 0 };
    MPI_Dims_create(nprocs[k], 2, dims2);
    MPI_Dims_create(nprocs[k], 3, dims3);
    printf("%3d processes: 2D %2d x %d, 3D %d x %d x %d\n",
           nprocs[k], dims2[0], dims2[1], dims3[0], dims3[1], dims3[2]);
}

int fixed[2] = { 0, 2 };                        // 2 processes in dimension 1, MPI chooses dimension 0
MPI_Dims_create(12, 2, fixed);
printf(" 12 processes with dims[1] = 2: %d x %d\n", fixed[0], fixed[1]);

### Coordinates and Ranks

| Call | Question |
|---|---|
| `MPI_Comm_rank(comm_cart, &rank)` | who am I in the grid? |
| `MPI_Cart_coords(comm_cart, rank, maxdims, coords)` | where is `rank` in the grid? |
| `MPI_Cart_rank(comm_cart, coords, &rank)` | who is at `coords`? In periodic dimensions, coordinates out of range wrap around; in non-periodic ones they are an error |
| `MPI_Cart_get(comm_cart, maxdims, dims, periods, coords)` | the dimensions, periods and my own coordinates |

The following cell creates the grid of the figure and prints it on rank 0, collected with `MPI_Gather()` — with the top row (the largest coordinate in dimension 1) first, as in the figure.

In [ ]:
%%mpi -n 12 -o code/topo-cart-create.c
if (size != 12) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 12 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

// the dimensions of the process grid: dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { [DIM_I] = 4, [DIM_J] = 3 }, periods[NUM_DIMS] = { [DIM_I] = 1, [DIM_J] = 0 };
MPI_Comm cart;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &cart);   // reorder = 0

int crank, coords[NUM_DIMS];
MPI_Comm_rank(cart, &crank);
MPI_Cart_coords(cart, crank, NUM_DIMS, coords);

// the right neighbor of the last column: dimension 0 is periodic, so (4, c1) wraps around to (0, c1)
int beyond[NUM_DIMS] = { [DIM_I] = dims[DIM_I], [DIM_J] = coords[DIM_J] }, wrapped;
MPI_Cart_rank(cart, beyond, &wrapped);

// the fields every rank sends to rank 0, in this order
enum { RANK, COORD_I, COORD_J, WRAPPED, NUM_FIELDS };

int mine[NUM_FIELDS] = { [RANK] = crank, [COORD_I] = coords[DIM_I], [COORD_J] = coords[DIM_J], [WRAPPED] = wrapped };
int *all = crank == 0 ? malloc(NUM_FIELDS * size * sizeof(int)) : NULL;
MPI_Gather(mine, NUM_FIELDS, MPI_INT, all, NUM_FIELDS, MPI_INT, 0, cart);

if (crank == 0) {
    for (int c1 = dims[DIM_J] - 1; c1 >= 0; --c1) {
        for (int c0 = 0; c0 < dims[DIM_I]; ++c0) {
            int c[NUM_DIMS] = { [DIM_I] = c0, [DIM_J] = c1 }, r;
            MPI_Cart_rank(cart, c, &r);
            const int *f = &all[NUM_FIELDS * r];        // the fields of rank r
            printf("   %2d (%d,%d)", f[RANK], f[COORD_I], f[COORD_J]);
        }
        printf("        coordinates (4,%d) wrap around to rank %d\n", c1, all[NUM_FIELDS * c1 + WRAPPED]);
    }
    free(all);
}

MPI_Comm_free(&cart);

### Neighbors: `MPI_Cart_shift()`

```c
int MPI_Cart_shift(MPI_Comm comm_cart, int direction, int disp, int *rank_source, int *rank_dest);
```

is a **local query**: it returns the ranks of the two processes at distance `disp` from the calling process in dimension `direction` — `rank_source` in negative, `rank_dest` in positive direction.
It does not communicate and moves no data.

![What MPI_Cart_shift returns in the 4 by 3 grid with periodic dimension 0. Inner rank 4 = (1,1): in dimension 0, src is 1 and dst is 7; in dimension 1, src is 3 and dst is 5. Boundary rank 11 = (3,2): in dimension 0, src is 8 and dst wraps around to rank 2; in dimension 1, src is 10 and dst is MPI_PROC_NULL. Below, rank 4 in dimension 0 with different displacements: disp 1 gives src 1 and dst 7, disp -1 swaps them, disp 2 gives src and dst 10 (wrap-around). In a non-periodic dimension, a neighbor beyond the boundary is MPI_PROC_NULL](img/cart-shift.svg)

* **Neighbors:** the most common use. With `disp = 1`, one call per dimension returns both neighbors — left and right, bottom and top — as in the ghost cell exchange below.
* **Boundaries:** in a periodic dimension, the coordinates wrap around; in a non-periodic one, a neighbor beyond the boundary is `MPI_PROC_NULL`. A send to or a receive from `MPI_PROC_NULL` does nothing, so the boundary ranks need no special cases.
* **Displacement:** `disp = -1` swaps source and dest. `disp = k` returns the ranks k steps away, e.g. for the initial alignment in Cannon's matrix multiplication, where every row of blocks is shifted by its own row index. `disp = 0` returns the calling rank twice.
* **The names** come from a shift along the dimension: if every rank sends to `rank_dest` and receives from `rank_source`, e.g. with `MPI_Sendrecv_replace()`, the data moves by `disp` — the ring shift and the open chain of [Point-to-Point Communication](point-to-point.ipynb), with the neighbors computed by MPI.
* **Diagonal neighbors**, e.g. for the corners of a 9-point stencil, are not available: compute their coordinates and get the rank with `MPI_Cart_rank()` — in non-periodic dimensions, check the range yourself.

The following cell prints `src` and `dst` of all ranks for `disp = 1`, those of rank 0 for other displacements, and finally uses `src` and `dst` to shift the values in dimension 1.

> 🤔 **Predict:** What are `src` and `dst` of ranks 0 and 8 in both dimensions? Which values do the ranks hold after the shift in dimension 1?

In [ ]:
%%mpi -n 12 -o code/topo-cart-shift.c
if (size != 12) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 12 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

// the dimensions of the process grid: dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { [DIM_I] = 4, [DIM_J] = 3 }, periods[NUM_DIMS] = { [DIM_I] = 1, [DIM_J] = 0 };
MPI_Comm cart;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &cart);
int crank;
MPI_Comm_rank(cart, &crank);

// a rank as text, "-" for MPI_PROC_NULL
#define NAME(r, buf) ((r) == MPI_PROC_NULL ? "-" : (snprintf(buf, sizeof(buf), "%d", (r)), buf))
char s[16], d[16];

// 1. the query: src and dst of every rank for disp = 1, collected on rank 0
enum { SRC, DST, NUM_PER_DIM };                 // the two results of MPI_Cart_shift() for one dimension
int mine[NUM_DIMS][NUM_PER_DIM];                       // mine[dim][SRC or DST]
MPI_Cart_shift(cart, DIM_I, 1, &mine[DIM_I][SRC], &mine[DIM_I][DST]);
MPI_Cart_shift(cart, DIM_J, 1, &mine[DIM_J][SRC], &mine[DIM_J][DST]);
int *all = crank == 0 ? malloc(NUM_DIMS * NUM_PER_DIM * size * sizeof(int)) : NULL;
MPI_Gather(mine, NUM_DIMS * NUM_PER_DIM, MPI_INT, all, NUM_DIMS * NUM_PER_DIM, MPI_INT, 0, cart);

if (crank == 0) {
    for (int dim = 0; dim < NUM_DIMS; ++dim) {
        printf("src > rank > dst in dimension %d (%s), top row first:\n", dim, periods[dim] ? "periodic" : "not periodic");
        for (int c1 = dims[DIM_J] - 1; c1 >= 0; --c1) {
            for (int c0 = 0; c0 < dims[DIM_I]; ++c0) {
                int c[NUM_DIMS] = { [DIM_I] = c0, [DIM_J] = c1 }, r;
                MPI_Cart_rank(cart, c, &r);
                const int *f = &all[(NUM_DIMS * r + dim) * NUM_PER_DIM];   // src and dst of rank r in dimension dim
                printf("   %2s >%2d > %-2s", NAME(f[SRC], s), r, NAME(f[DST], d));
            }
            printf("\n");
        }
    }

    // MPI_Cart_shift() is local: rank 0 alone can query other displacements
    int disps[4] = { -1, 1, 2, 4 }, src, dst;
    printf("rank 0 = (0,0) with other displacements:\n");
    for (int dim = 0; dim < NUM_DIMS; ++dim)
        for (int k = 0; k < 4; ++k) {
            MPI_Cart_shift(cart, dim, disps[k], &src, &dst);
            printf("  dim %d, disp %2d: src %s, dst %s\n", dim, disps[k], NAME(src, s), NAME(dst, d));
        }
}

// 2. using src and dst for a shift: every rank sends its value to dst and receives from src
int src, dst, value = crank;
MPI_Cart_shift(cart, DIM_J, 1, &src, &dst);
MPI_Sendrecv_replace(&value, 1, MPI_INT, dst, 0, src, 0, cart, MPI_STATUS_IGNORE);
MPI_Gather(&value, 1, MPI_INT, all, 1, MPI_INT, 0, cart);

if (crank == 0) {
    printf("values after the shift in dimension 1 (before: value = rank), top row first:\n");
    for (int c1 = dims[DIM_J] - 1; c1 >= 0; --c1) {
        for (int c0 = 0; c0 < dims[DIM_I]; ++c0) {
            int c[NUM_DIMS] = { [DIM_I] = c0, [DIM_J] = c1 }, r;
            MPI_Cart_rank(cart, c, &r);
            printf(" %3d", all[r]);
        }
        printf("\n");
    }
    free(all);
}

MPI_Comm_free(&cart);

`MPI_PROC_NULL` is a named constant; its numerical value (e.g. -2 in Open MPI) is implementation-specific — always compare with the name.

### Row and Column Communicators: `MPI_Cart_sub()`

Many operations in a domain decomposition are neither local nor global: they run **along one dimension** of the process grid.
Examples are a sum over every row, a broadcast down every column, or a transpose within every plane.
A collective always involves *all* processes of its communicator, so every row needs a communicator of its own.
You could work out the members of each row from the coordinates yourself. `MPI_Cart_sub()` does it for you, straight from the topology:

```c
int MPI_Cart_sub(MPI_Comm comm_cart, const int remain_dims[], MPI_Comm *newcomm);
```

splits a Cartesian grid into lower-dimensional sub-grids: the dimensions with `remain_dims[i] = 1` are kept, all others are fixed.

![The sub-communicators of MPI_Cart_sub in the 4 by 3 grid. Left, remain_dims = {1, 0}: dimension 0 is kept, so there are three communicators of 4 processes, one per value of c1 — the horizontal rows of the figure; the new rank is c0. Right, remain_dims = {0, 1}: dimension 1 is kept, so there are four communicators of 3 processes, one per value of c0 — the vertical columns; the new rank is c1. Below, a 3 by 3 by 2 grid: remain_dims = {1, 1, 0} gives two planes of 3 by 3 processes, remain_dims = {0, 0, 1} gives nine lines of 2 processes](img/cart-sub.svg)

* **Which processes:** the kept dimensions vary inside a sub-communicator, the dropped ones are fixed. There is one sub-communicator per combination of the dropped coordinates, and its size is the product of the kept `dims`.
  In the 4 × 3 grid, keeping dimension 0 gives three communicators with 4 processes each — one per row of the figure, as dimension 0 is drawn horizontally; keeping dimension 1 gives four column communicators with 3 processes each.
  (The Communicators notebook uses the matrix convention, with dimension 0 as the row index — there, the same communicators are called the other way round.)
* **New ranks:** the processes of a sub-communicator are numbered by their kept coordinates, in the same order as in `comm_cart`; with one kept dimension, the new rank is the coordinate in that dimension.
* **Still a grid:** the new communicator has a Cartesian topology with the kept `dims` and `periods` — `MPI_Cart_coords()`, `MPI_Cart_shift()` and the neighborhood collectives work in it.
* **Collective:** all processes of `comm_cart` call `MPI_Cart_sub()`, and every process belongs to exactly one of the new communicators. It is the Cartesian form of `MPI_Comm_split()` (see Communicators), with the dropped coordinates as color and the kept ones as key.

> 🤔 **Predict:** Which rank does rank 7 have in its row and in its column communicator? What are the sums of the ranks in both?

In [ ]:
%%mpi -n 12 -o code/topo-cart-sub.c
if (size != 12) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 12 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

// the dimensions of the process grid: dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { [DIM_I] = 4, [DIM_J] = 3 }, periods[NUM_DIMS] = { 0, 0 };
MPI_Comm cart, rowcomm, colcomm;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &cart);
int crank, coords[NUM_DIMS];
MPI_Comm_rank(cart, &crank);
MPI_Cart_coords(cart, crank, NUM_DIMS, coords);

int keep_dim0[NUM_DIMS] = { [DIM_I] = 1, [DIM_J] = 0 }, keep_dim1[NUM_DIMS] = { [DIM_I] = 0, [DIM_J] = 1 };
MPI_Cart_sub(cart, keep_dim0, &rowcomm);    // the processes with the same coordinate in dimension 1
MPI_Cart_sub(cart, keep_dim1, &colcomm);    // the processes with the same coordinate in dimension 0

int rowrank, rowsize, colrank, colsize, rowsum, colsum;
MPI_Comm_rank(rowcomm, &rowrank);
MPI_Comm_size(rowcomm, &rowsize);
MPI_Comm_rank(colcomm, &colrank);
MPI_Comm_size(colcomm, &colsize);
MPI_Allreduce(&crank, &rowsum, 1, MPI_INT, MPI_SUM, rowcomm);  // sum of the ranks in my row
MPI_Allreduce(&crank, &colsum, 1, MPI_INT, MPI_SUM, colcomm);  // sum of the ranks in my column

// the fields every rank sends to rank 0, in this order
enum { COORD_I, COORD_J, ROWRANK, ROWSIZE, ROWSUM, COLRANK, COLSIZE, COLSUM, NUM_FIELDS };

int mine[NUM_FIELDS] = { [COORD_I]  = coords[DIM_I], [COORD_J]  = coords[DIM_J],
                         [ROWRANK] = rowrank,   [ROWSIZE] = rowsize,   [ROWSUM] = rowsum,
                         [COLRANK] = colrank,   [COLSIZE] = colsize,   [COLSUM] = colsum };
int *all = crank == 0 ? malloc(NUM_FIELDS * size * sizeof(int)) : NULL;
MPI_Gather(mine, NUM_FIELDS, MPI_INT, all, NUM_FIELDS, MPI_INT, 0, cart);
if (crank == 0) {
    for (int r = 0; r < size; ++r) {
        const int *f = &all[NUM_FIELDS * r];    // the fields of rank r
        printf("rank %2d (%d,%d): rank %d of %d in its row (sum %2d), rank %d of %d in its column (sum %2d)\n",
               r, f[COORD_I], f[COORD_J], f[ROWRANK], f[ROWSIZE], f[ROWSUM], f[COLRANK], f[COLSIZE], f[COLSUM]);
    }
    free(all);

    // the sub-communicators are Cartesian again: 1D, with the kept dims and periods
    MPI_Comm sub[2] = { rowcomm, colcomm };
    const char *name[2] = { "rowcomm", "colcomm" };
    for (int k = 0; k < 2; ++k) {
        int ndims, d, p, c;
        MPI_Cartdim_get(sub[k], &ndims);
        MPI_Cart_get(sub[k], 1, &d, &p, &c);
        printf("%s of rank 0: %dD Cartesian, dims %d, periods %d, my coordinate %d\n", name[k], ndims, d, p, c);
    }
}

MPI_Comm_free(&rowcomm);
MPI_Comm_free(&colcomm);
MPI_Comm_free(&cart);

#### Why Sub-Communicators Pay Off

Why not do everything on the whole grid communicator? A sub-communicator brings four advantages:

* **Smaller collectives, all at the same time.** On a 32 × 32 grid of 1024 processes, a reduction along the rows is 32 *independent* allreduces of 32 processes each, and they run at the same time. A tree over 32 processes has 5 levels, a tree over 1024 has 10. The messages also stay within the row; with a good mapping of the grid onto the machine (`reorder = 1`), a row spans only a few nodes.
* **Less data.** Many 2D algorithms need a block only in its row or its column. In the SUMMA matrix multiplication on a $q \times q$ grid, every process receives $2n^2/q$ matrix elements in total. A 1D distribution, by contrast, needs all of $B$ on every process: $n^2$ elements, $q/2 = \sqrt{P}/2$ times as many.
* **No rank arithmetic.** Nobody computes who belongs to which row, and the code stays correct with `reorder = 1`, where the ranks in `MPI_COMM_WORLD` no longer follow the grid. Because the new rank equals the kept coordinate, the root of a row broadcast is simply the column coordinate of the sender.
* **The structure is kept.** A row communicator is a 1D Cartesian communicator with the period of its dimension, so `MPI_Cart_shift()` inside it gives the neighbors in the row — e.g. for a ring algorithm such as Cannon's matrix multiplication.

Like every communicator constructor, `MPI_Cart_sub()` is a collective call that allocates a new context (Splitting Is Not Free): create the sub-communicators **once** after the grid, not in every iteration.

#### Use Cases

| Application | Process grid | Sub-communicators | Communication in them |
|---|---|---|---|
| block matrix–vector product $y = Ax$ (Communicators) | 2D | rows and columns | `MPI_Allreduce()` of the partial products $A_{ij}x_j$ along the rows; distribution of $x_j$ down the columns |
| SUMMA matrix multiplication $C = AB$ (ScaLAPACK, Elemental) | 2D, $q \times q$ | rows and columns | in step $k$: `MPI_Bcast()` of the $k$-th block column of $A$ along every row, of the $k$-th block row of $B$ down every column |
| 3D FFT with a **pencil** decomposition (P3DFFT, 2DECOMP&FFT, heFFTe) | 2D grid over an $N^3$ array | rows and columns | two `MPI_Alltoall()` transposes, one in the rows, one in the columns — up to $N^2$ processes instead of $N$ with slabs |
| line solvers: ADI, line relaxation, tridiagonal systems along grid lines | 2D or 3D | the lines in the solve direction | pipelined or parallel-cyclic-reduction solve along every line |
| statistics over homogeneous directions, e.g. the mean velocity profile of a turbulent channel flow | 3D, wall-normal dimension 1 | planes `{1, 0, 1}` | `MPI_Allreduce()` per plane: one average per wall distance |
| global offsets of tiles with different sizes (load balancing, parallel I/O) | 2D or 3D | lines in every dimension | `MPI_Exscan()` of the local sizes along each dimension |
| ensembles and parameter studies | 3D: members × 2D domain | `{0, 1, 1}` per member, `{1, 0, 0}` across members | each member runs its own simulation; `MPI_Allreduce()` across members for the ensemble mean |
| training of large language models (LLM Parallelism) | 3D: DP × PP × TP, e.g. 128 × 16 × 8 for Llama 3 405B | one dimension each | gradient allreduce in the DP lines (128 processes), allreduce of partial results in the TP lines (8 GPUs of a node), send/receive between neighboring pipeline stages (`MPI_Cart_shift()` in the PP dimension) |

Two sketches with the row and column communicators of the cell above — `rowcomm`: fixed `c1`, rank = `c0`; `colcomm`: fixed `c0`, rank = `c1`.
The **global start index** of tiles of different sizes is a prefix sum along each dimension:

```c
int i0, j0;                                                    // global index of my first inner cell
MPI_Exscan(&imaxLocal, &i0, 1, MPI_INT, MPI_SUM, rowcomm);     // sum of the tiles to my left
MPI_Exscan(&jmaxLocal, &j0, 1, MPI_INT, MPI_SUM, colcomm);     // sum of the tiles below me
if (coords[DIM_I] == 0) i0 = 0;                                    // MPI_Exscan leaves rank 0 undefined
if (coords[DIM_J] == 0) j0 = 0;
```

One step of **SUMMA** on a $q \times q$ grid, where process (c0, c1) holds the blocks of block row c1 and block column c0:

```c
for (int k = 0; k < q; ++k) {
    if (coords[DIM_I] == k) memcpy(Abuf, A, nb * nb * sizeof(double));   // column k provides its blocks of A ...
    MPI_Bcast(Abuf, nb * nb, MPI_DOUBLE, k, rowcomm);                  // ... to its row: root k is c0 = k
    if (coords[DIM_J] == k) memcpy(Bbuf, B, nb * nb * sizeof(double));   // row k provides its blocks of B ...
    MPI_Bcast(Bbuf, nb * nb, MPI_DOUBLE, k, colcomm);                  // ... to its column: root k is c1 = k
    gemm(nb, Abuf, Bbuf, C);                                           // C += Abuf · Bbuf
}
```

All rows broadcast at the same time, and so do all columns. Each process takes part in only $2q$ broadcasts of $q$ processes each.

## Putting It All Together: 2D Ghost Cell Exchange

In a 2D domain decomposition, every tile exchanges ghost cells with up to four neighbors, and the left and right boundary layers are **columns** — a job for [derived types](datatypes.ipynb):

![The local grid of a rank: inner cells surrounded by one layer of ghost cells. The ghost cells receive the boundary layers of the neighbors; the boundary layer of the inner cells is sent. A row is a contiguous type of imaxLocal doubles, a column a vector type with jmaxLocal blocks of one double and a stride of imaxLocal + 2. The corner ghost cells are not exchanged](img/halo-types.svg)

The local grid has `(imaxLocal + 2) × (jmaxLocal + 2)` cells, stored row by row: `grid[j * (imaxLocal + 2) + i]`.
Dimension 0 of the process grid runs along `i` (left/right), dimension 1 along `j` (bottom/top).
The steps are

1. set up the process grid with `MPI_Dims_create()` and `MPI_Cart_create()`, and the neighbors with `MPI_Cart_shift()`,
2. compute the size of the local tile **from the coordinates** of the rank, not from the rank itself, which only works in 1D (see *Why the coordinates, not the rank?* below),
3. create the row type (`MPI_Type_contiguous()`) and the column type (`MPI_Type_vector()`),
4. exchange: receive into each ghost layer from the neighbor on that side, and send the adjacent boundary layer to it.

Left and right neighbors have the same coordinate in dimension 1, and thus the same `jmaxLocal`; bottom and top neighbors have the same `imaxLocal`.
So the column and row types match on both sides, even if the tiles have different sizes.

Every direction of travel gets its own **tag**.
With identical tags, messages from the same neighbor could be mixed up — which happens as soon as the left and the right neighbor are the same process, e.g. in a periodic dimension with 2 processes.

### Debugging the Exchange

The best way to check an exchange is a small grid in which every rank fills its own cells with its rank number and its ghost cells with -1: after one exchange, every ghost cell must show the rank of the neighbor on that side.
Printing from all ranks in turn with barriers does not reliably produce ordered output: the output of every rank is forwarded to the terminal independently.
Instead, the helper `print_grid()` collects all tiles on rank 0 — each received **directly into its place** in a large array with a subarray type — and prints them side by side, top row first.
The following cell writes it to a header file, which the exchange cells include with `-i dt-print-grid.h -f Icode`:

In [ ]:
%%writefile code/topo-print-grid.h
// Debugging helper for 2D domain decompositions: collects the local grids of all ranks of a
// 2D Cartesian communicator on rank 0 and prints them side by side, ghost cells included,
// with the top row first. The local grid has (imaxLocal + 2) x (jmaxLocal + 2) cells, stored
// row by row: grid[j * (imaxLocal + 2) + i]. Dimension 0 of the communicator runs along i,
// dimension 1 along j. Uses the tag 9999.
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>

static void print_grid(MPI_Comm comm, const double *grid, int imaxLocal, int jmaxLocal, int width, int precision) {
    // the dimensions of the process grid: dimension 0 runs along i, dimension 1 along j
    enum { DIM_I, DIM_J, NUM_DIMS };
    int rank, size, dims[NUM_DIMS], periods[NUM_DIMS], coords[NUM_DIMS];
    MPI_Comm_rank(comm, &rank);
    MPI_Comm_size(comm, &size);
    MPI_Cart_get(comm, NUM_DIMS, dims, periods, coords);

    // rank 0 needs the size of every tile
    int local[NUM_DIMS] = { [DIM_I] = imaxLocal + 2, [DIM_J] = jmaxLocal + 2 }, *tiles = NULL;
    if (rank == 0)
        tiles = malloc(NUM_DIMS * size * sizeof(int));
    MPI_Gather(local, NUM_DIMS, MPI_INT, tiles, NUM_DIMS, MPI_INT, 0, comm);

    MPI_Request request;
    MPI_Isend(grid, local[DIM_I] * local[DIM_J], MPI_DOUBLE, 0, 9999, comm, &request);

    if (rank == 0) {
        // the width of every column and the height of every row of tiles
        int *wi = calloc(dims[DIM_I], sizeof(int)), *hj = calloc(dims[DIM_J], sizeof(int));
        for (int r = 0; r < size; ++r) {
            int c[NUM_DIMS];
            MPI_Cart_coords(comm, r, NUM_DIMS, c);
            wi[c[DIM_I]] = tiles[NUM_DIMS * r + DIM_I];
            hj[c[DIM_J]] = tiles[NUM_DIMS * r + DIM_J];
        }

        // the canvas holds all tiles with a gap of one cell in between; NAN marks the gaps
        int W = dims[DIM_I] - 1, H = dims[DIM_J] - 1;
        for (int k = 0; k < dims[DIM_I]; ++k) W += wi[k];
        for (int k = 0; k < dims[DIM_J]; ++k) H += hj[k];
        double *canvas = malloc((size_t)W * H * sizeof(double));
        for (int k = 0; k < W * H; ++k)
            canvas[k] = NAN;

        // receive every tile directly into its place in the canvas, with a subarray type
        for (int r = 0; r < size; ++r) {
            int c[NUM_DIMS], oi = 0, oj = 0;
            MPI_Cart_coords(comm, r, NUM_DIMS, c);
            for (int k = 0; k < c[DIM_I]; ++k) oi += wi[k] + 1;
            for (int k = 0; k < c[DIM_J]; ++k) oj += hj[k] + 1;

            int sizes[2] = { H, W }, subsizes[2] = { hj[c[DIM_J]], wi[c[DIM_I]] }, starts[2] = { oj, oi };
            MPI_Datatype tile;
            MPI_Type_create_subarray(2, sizes, subsizes, starts, MPI_ORDER_C, MPI_DOUBLE, &tile);
            MPI_Type_commit(&tile);
            MPI_Recv(canvas, 1, tile, r, 9999, comm, MPI_STATUS_IGNORE);
            MPI_Type_free(&tile);
        }

        for (int j = H - 1; j >= 0; --j) {
            for (int i = 0; i < W; ++i) {
                double v = canvas[(size_t)j * W + i];
                if (isnan(v))
                    printf("%*s", width + 1, "");
                else
                    printf(" %*.*f", width, precision, v);
            }
            printf("\n");
        }
        free(canvas);
        free(wi);
        free(hj);
        free(tiles);
    }
    MPI_Wait(&request, MPI_STATUS_IGNORE);
}

Here is the complete exchange with nonblocking point-to-point calls, on a 8 × 6 grid distributed to 6 processes.

> 🤔 **Predict:** Which values do the ghost cells of the tile in the middle of the bottom row show after the exchange? Which ghost cells keep the value -1?

In [ ]:
%%mpi -n 6 -i topo-print-grid.h -f Icode -o code/topo-halo-exchange.c
// the global grid: imax x jmax inner cells
const int imax = 8, jmax = 6;

// 2D process grid, not periodic; dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { 0, 0 }, periods[NUM_DIMS] = { 0, 0 }, coords[NUM_DIMS];
MPI_Dims_create(size, NUM_DIMS, dims);
MPI_Comm comm;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &comm);
MPI_Comm_rank(comm, &rank);
MPI_Cart_coords(comm, rank, NUM_DIMS, coords);

// the neighbors: MPI_PROC_NULL at the boundary of the global grid
int left, right, bottom, top;
MPI_Cart_shift(comm, DIM_I, 1, &left, &right);
MPI_Cart_shift(comm, DIM_J, 1, &bottom, &top);

// the size of the local tile - computed from the coordinates, not from the rank!
#define SIZE_OF(c, n, p) ((n) / (p) + ((c) < (n) % (p) ? 1 : 0))
const int imaxLocal = SIZE_OF(coords[DIM_I], imax, dims[DIM_I]);
const int jmaxLocal = SIZE_OF(coords[DIM_J], jmax, dims[DIM_J]);
const int ld = imaxLocal + 2;                   // leading dimension: length of a row incl. ghost cells = stride from row j to row j + 1

// the local grid with one layer of ghost cells, stored row by row
double *grid = malloc((size_t)ld * (jmaxLocal + 2) * sizeof(double));
#define G(i, j) grid[(j) * ld + (i)]

// derived types for the ghost layers
MPI_Datatype column, row;
MPI_Type_vector(jmaxLocal, 1, ld, MPI_DOUBLE, &column);     // left and right
MPI_Type_contiguous(imaxLocal, MPI_DOUBLE, &row);           // bottom and top
MPI_Type_commit(&column);
MPI_Type_commit(&row);

// for debugging: own cells hold the rank, ghost cells -1
for (int j = 0; j < jmaxLocal + 2; ++j)
    for (int i = 0; i < ld; ++i)
        G(i, j) = (i == 0 || i == ld - 1 || j == 0 || j == jmaxLocal + 1) ? -1 : rank;

// the tags: the direction of travel of a message
enum { UPWARD, DOWNWARD, RIGHTWARD, LEFTWARD };

// the requests: one receive from and one send to every neighbor
enum { RECV_BOTTOM, RECV_TOP, RECV_LEFT, RECV_RIGHT,
       SEND_TOP, SEND_BOTTOM, SEND_RIGHT, SEND_LEFT, NUM_REQUESTS };

// exchange: receive into the ghost layers, send the adjacent boundary layers
MPI_Request requests[NUM_REQUESTS];
MPI_Irecv(&G(1, 0),             1, row,    bottom, UPWARD,    comm, &requests[RECV_BOTTOM]);
MPI_Irecv(&G(1, jmaxLocal + 1), 1, row,    top,    DOWNWARD,  comm, &requests[RECV_TOP]);
MPI_Irecv(&G(0, 1),             1, column, left,   RIGHTWARD, comm, &requests[RECV_LEFT]);
MPI_Irecv(&G(imaxLocal + 1, 1), 1, column, right,  LEFTWARD,  comm, &requests[RECV_RIGHT]);
MPI_Isend(&G(1, jmaxLocal),     1, row,    top,    UPWARD,    comm, &requests[SEND_TOP]);
MPI_Isend(&G(1, 1),             1, row,    bottom, DOWNWARD,  comm, &requests[SEND_BOTTOM]);
MPI_Isend(&G(imaxLocal, 1),     1, column, right,  RIGHTWARD, comm, &requests[SEND_RIGHT]);
MPI_Isend(&G(1, 1),             1, column, left,   LEFTWARD,  comm, &requests[SEND_LEFT]);
MPI_Waitall(NUM_REQUESTS, requests, MPI_STATUSES_IGNORE);

print_grid(comm, grid, imaxLocal, jmaxLocal, 2, 0);

MPI_Type_free(&column);
MPI_Type_free(&row);
MPI_Comm_free(&comm);
free(grid);

Every tile shows its rank in its own cells and the ranks of its neighbors in its ghost cells; -1 remains at the global boundary — where the neighbor is `MPI_PROC_NULL` — and in the corners.
Try other numbers of processes, e.g. `-n 4` or `-n 3`, and check that the tiles still fit.

**Why the coordinates, not the rank?**
In 1D, rank and coordinate are the same, and the size of a tile is often computed from the rank, e.g. with `sizeOfRank(rank, size, n) = n / size + (rank < n % size ? 1 : 0)`.
Carried over to 2D as

```c
imaxLocal = sizeOfRank(rank, dims[DIM_I], imax);
jmaxLocal = sizeOfRank(rank, dims[DIM_J], jmax);
```

this is wrong.
A 2D decomposition consists of two 1D decompositions: `imax` is split among the `dims[DIM_I]` columns of tiles, `jmax` among the `dims[DIM_J]` rows of tiles.
So the width of a tile may only depend on its coordinate `c0`, and its height only on `c1`.
The rank `c0 · dims[1] + c1` mixes both coordinates, and it runs up to `size - 1` instead of `dims[d] - 1`.
For the 8 × 6 grid on 3 × 2 processes above:

| Ranks | Coordinates | `imaxLocal` from `c0` | `imaxLocal` from the rank |
|---|---|---|---|
| 0, 1 | (0, 0), (0, 1) | 3 | 3 |
| 2, 3 | (1, 0), (1, 1) | 3 | **2** |
| 4, 5 | (2, 0), (2, 1) | 2 | 2 |

The tiles are only 3 + 2 + 2 = 7 cells wide in total, so one column of the global grid is lost.
Still, the exchange runs without any error and the output looks plausible: by chance, the tiles of each column agree with each other, and `jmaxLocal` is right only because 6 is divisible by 2.
With a 7 × 7 grid, rank 0 gets a 3 × 4 tile and all other ranks a 2 × 3 tile, so neighbors disagree.
Rank 0 sends a row of 3 values to its top neighbor (rank 1) and a column of 4 values to its right neighbor (rank 2), which expect only 2 and 3 values.
The program aborts with `MPI_ERR_TRUNCATE`.
The messages in the opposite direction are shorter than expected, which is not an error: they would silently leave ghost cells of rank 0 stale.

The coordinates guarantee what the exchange relies on:

* bottom and top neighbors share `c0` and thus `imaxLocal`, so their row types match,
* left and right neighbors share `c1` and thus `jmaxLocal`, so their column types match,
* the tiles add up to the global grid.

With `reorder = 1`, the rank wouldn't even tell where a tile is.
The function itself is fine, only its argument is wrong: `sizeOfRank(coords[DIM_I], dims[DIM_I], imax)` is exactly `SIZE_OF(coords[DIM_I], imax, dims[DIM_I])`.

> 🤔 **Try it:** in the cell above, replace `coords[DIM_I]` and `coords[DIM_J]` with `rank` in the two `SIZE_OF()` lines, run the cell and count the columns of the global grid. Then set `imax = jmax = 7`.

Note that `G(1, 1)` is the start of two send buffers (the bottom row and the left column) at the same time.
That's fine: pending sends may share — and read — the same memory.

## Neighborhood Collectives

Since MPI 3.0, the communication with all neighbors in a virtual topology can be expressed as a single collective operation — a **neighborhood collective**.

* It must be called by all processes of the communicator, like any collective.
* The MPI library knows the complete pattern and can optimize it, e.g. for the network topology.
* For a Cartesian topology, the **order of the neighbors** in the buffers is fixed: dimension by dimension, first the neighbor in negative, then in positive direction — the `rank_source` and `rank_dest` of `MPI_Cart_shift(comm, d, 1, …)`.
  A rank always has `2 · ndims` neighbors; the entries for `MPI_PROC_NULL` neighbors are not touched.

![The four neighbors of a rank in a 2D Cartesian grid and their positions in the receive buffer: 0 is the neighbor in negative direction of dimension 0 (left), 1 in positive direction (right), 2 in negative direction of dimension 1 (bottom), 3 in positive direction (top)](img/neighbor-order.svg)

### Neighbor Allgather

```c
int MPI_Neighbor_allgather(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                           void *recvbuf, int recvcount, MPI_Datatype recvtype, MPI_Comm comm);
```

sends the same data to all neighbors and receives `recvcount` elements from each of them, stored one after the other in the neighbor order.
`MPI_Neighbor_allgatherv()` allows a different count and displacement for every neighbor.

![MPI_Neighbor_allgather for rank 8 = (2,2) of the 4 by 3 grid, periodic in dimension 0: ranks 5 (left), 11 (right) and 7 (bottom) each send their one-element sendbuf to rank 8; the top neighbor is MPI_PROC_NULL. The recvbuf of rank 8 holds 5, 11, 7 in slots 0 to 2; slot 3 is not touched and keeps -1. Rank 8 sends its own 8 to ranks 5, 11 and 7, where it lands in slots 1, 0 and 3](img/neighbor-allgather.svg)

Note the direction: slot `i` holds the block that came **from** neighbor `i`. My own block therefore lands in the *opposite* slot of each neighbor — in slot 1 ("from the right") of my left neighbor.

> 🤔 **Predict:** In the 4 × 3 grid, periodic in dimension 0, what are the neighbors of rank 0? Of rank 4?

In [ ]:
%%mpi -n 12 -o code/topo-neighbor-allgather.c
if (size != 12) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 12 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

// the dimensions of the process grid: dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { [DIM_I] = 4, [DIM_J] = 3 }, periods[NUM_DIMS] = { [DIM_I] = 1, [DIM_J] = 0 };
MPI_Comm cart;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &cart);
int crank;
MPI_Comm_rank(cart, &crank);

// the neighbor order of a Cartesian topology: for each dimension the - and then the + neighbor,
// i.e. the order of the results of MPI_Cart_shift() for dimension 0, then for dimension 1
enum { LEFT, RIGHT, BOTTOM, TOP, NUM_NEIGHBORS };

// every rank sends its rank to all neighbors; entries of MPI_PROC_NULL neighbors stay -1
int neighbors[NUM_NEIGHBORS] = { [LEFT] = -1, [RIGHT] = -1, [BOTTOM] = -1, [TOP] = -1 };
MPI_Neighbor_allgather(&crank, 1, MPI_INT, neighbors, 1, MPI_INT, cart);

int *all = crank == 0 ? malloc(NUM_NEIGHBORS * size * sizeof(int)) : NULL;
MPI_Gather(neighbors, NUM_NEIGHBORS, MPI_INT, all, NUM_NEIGHBORS, MPI_INT, 0, cart);
if (crank == 0) {
    for (int r = 0; r < size; ++r) {
        int c[NUM_DIMS];
        MPI_Cart_coords(cart, r, NUM_DIMS, c);
        const int *f = &all[NUM_NEIGHBORS * r];     // the neighbors of rank r
        printf("rank %2d (%d,%d): left %2d, right %2d, bottom %2d, top %2d\n",
               r, c[DIM_I], c[DIM_J], f[LEFT], f[RIGHT], f[BOTTOM], f[TOP]);
    }
    free(all);
}

MPI_Comm_free(&cart);

### Neighbor Alltoall

```c
int MPI_Neighbor_alltoall(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                          void *recvbuf, int recvcount, MPI_Datatype recvtype, MPI_Comm comm);
int MPI_Neighbor_alltoallv(const void *sendbuf, const int sendcounts[], const int sdispls[], MPI_Datatype sendtype,
                           void *recvbuf, const int recvcounts[], const int rdispls[], MPI_Datatype recvtype,
                           MPI_Comm comm);
int MPI_Neighbor_alltoallw(const void *sendbuf, const int sendcounts[], const MPI_Aint sdispls[],
                           const MPI_Datatype sendtypes[],
                           void *recvbuf, const int recvcounts[], const MPI_Aint rdispls[],
                           const MPI_Datatype recvtypes[], MPI_Comm comm);
```

* `MPI_Neighbor_alltoall()` sends a **different** chunk to every neighbor: chunk `i` of `sendbuf` goes to neighbor `i`, and the chunk from neighbor `i` lands at position `i` of `recvbuf`.
* `MPI_Neighbor_alltoallv()` allows different counts and displacements (in elements) for every neighbor.
* `MPI_Neighbor_alltoallw()` additionally allows a **different data type** for every neighbor, with displacements **in bytes**.

![MPI_Neighbor_alltoall for rank 8 = (2,2). Send: the sendbuf of rank 8 has four blocks 8 to 5, 8 to 11, 8 to 7 and 8 to top; block i goes to neighbor i, rank 5 stores its block in slot 1, rank 11 in slot 0, rank 7 in slot 3; block 3 is not sent, as the top neighbor is MPI_PROC_NULL. Receive: the recvbuf of rank 8 holds 5 to 8 (block 1 of rank 5), 11 to 8 (block 0 of rank 11), 7 to 8 (block 3 of rank 7), and slot 3 is not touched](img/neighbor-alltoall.svg)

The `v` and `w` variants cut the buffers into blocks exactly like `MPI_Alltoallv()` and `MPI_Alltoallw()` — see the figure in [Collective Communication](collectives.ipynb) — with one entry of `counts`, `displs` (and `types`) per neighbor instead of per rank.

The `w` variant is exactly what the ghost cell exchange needs: column types for the left and right neighbors, row types for the bottom and top neighbors, and displacements that point to the boundary and ghost layers of the local grid.

### Ghost Cell Exchange with `MPI_Neighbor_alltoallw()`

The eight point-to-point calls and the `MPI_Waitall()` become one collective call.
The send and receive buffer is the local grid in both cases; the displacements select the layers.
The arrays have one entry per neighbor, in the order of the results of `MPI_Cart_shift()`: for each dimension the − and then the + neighbor, i.e. left, right, bottom, top.
The code names these indices with an `enum` and fills the arrays with designated initializers such as `[LEFT] = column`, so that every entry shows which neighbor it belongs to.
The regions that are read and written are disjoint, so the same buffer is used for both.
This is common practice and works with the usual MPI implementations — but strictly speaking, it is not standard-conforming: the MPI standard forbids aliasing an output argument (here `recvbuf`) with any other argument, even if the regions actually accessed are disjoint.

![MPI_Neighbor_alltoallw as a ghost cell exchange. The local grid of me with ghost cells: the send regions s0 (first own column), s1 (last own column), s2 (first own row), s3 (last own row) and the receive regions r0 (left ghost column), r1 (right), r2 (bottom), r3 (top). My chunk 1, the column s1, lands in slot 0 of the right neighbor, its left ghost column r0; its chunk 0, its first own column s0, lands in my slot 1, the right ghost column r1. The displacements count in bytes from G(0,0); s0 and s2 both start at G(1,1) with different types](img/neighbor-alltoallw.svg)

| Chunk | Index | Type | Send region, `sdispls[i]` | Receive region, `rdispls[i]` |
|---|---|---|---|---|
| 0 | `LEFT` | `column` | first own column, `G(1, 1)` | left ghost column, `G(0, 1)` |
| 1 | `RIGHT` | `column` | last own column, `G(imaxLocal, 1)` | right ghost column, `G(imaxLocal + 1, 1)` |
| 2 | `BOTTOM` | `row` | first own row, `G(1, 1)` | bottom ghost row, `G(1, 0)` |
| 3 | `TOP` | `row` | last own row, `G(1, jmaxLocal)` | top ghost row, `G(1, jmaxLocal + 1)` |

The displacement of `G(i, j)` is `(j * ld + i) * sizeof(double)` bytes from the start of the grid, with the leading dimension `ld = imaxLocal + 2` — the macro `DISP(i, j)` in the code.

> 🤔 **Predict:** Will the result differ from the point-to-point version above?

In [ ]:
%%mpi -n 6 -i topo-print-grid.h -f Icode -o code/topo-halo-neighbor.c
// the global grid: imax x jmax inner cells
const int imax = 8, jmax = 6;

// 2D process grid, not periodic; dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { 0, 0 }, periods[NUM_DIMS] = { 0, 0 }, coords[NUM_DIMS];
MPI_Dims_create(size, NUM_DIMS, dims);
MPI_Comm comm;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &comm);
MPI_Comm_rank(comm, &rank);
MPI_Cart_coords(comm, rank, NUM_DIMS, coords);

// the neighbors: MPI_PROC_NULL at the boundary of the global grid
int left, right, bottom, top;
MPI_Cart_shift(comm, DIM_I, 1, &left, &right);
MPI_Cart_shift(comm, DIM_J, 1, &bottom, &top);

// the size of the local tile - computed from the coordinates, not from the rank!
#define SIZE_OF(c, n, p) ((n) / (p) + ((c) < (n) % (p) ? 1 : 0))
const int imaxLocal = SIZE_OF(coords[DIM_I], imax, dims[DIM_I]);
const int jmaxLocal = SIZE_OF(coords[DIM_J], jmax, dims[DIM_J]);
const int ld = imaxLocal + 2;                   // leading dimension: length of a row incl. ghost cells = stride from row j to row j + 1

// the local grid with one layer of ghost cells, stored row by row
double *grid = malloc((size_t)ld * (jmaxLocal + 2) * sizeof(double));
#define G(i, j) grid[(j) * ld + (i)]

// derived types for the ghost layers
MPI_Datatype column, row;
MPI_Type_vector(jmaxLocal, 1, ld, MPI_DOUBLE, &column);     // left and right
MPI_Type_contiguous(imaxLocal, MPI_DOUBLE, &row);           // bottom and top
MPI_Type_commit(&column);
MPI_Type_commit(&row);

// for debugging: own cells hold the rank, ghost cells -1
for (int j = 0; j < jmaxLocal + 2; ++j)
    for (int i = 0; i < ld; ++i)
        G(i, j) = (i == 0 || i == ld - 1 || j == 0 || j == jmaxLocal + 1) ? -1 : rank;

// the neighbor order of a Cartesian topology: for each dimension the - and then the + neighbor,
// i.e. the order of the results of MPI_Cart_shift() for dimension 0, then for dimension 1
enum { LEFT, RIGHT, BOTTOM, TOP, NUM_NEIGHBORS };

    // the displacement of G(i, j) in bytes from the start of the grid
    #define DISP(i, j) (((MPI_Aint)(j) * ld + (i)) * (MPI_Aint)sizeof(double))
    
    // one entry per neighbor, filled by name
    int          counts[NUM_NEIGHBORS]  = { [LEFT]   = 1,      [RIGHT] = 1,
                                            [BOTTOM] = 1,      [TOP]   = 1 };
    MPI_Datatype types[NUM_NEIGHBORS]   = { [LEFT]   = column, [RIGHT] = column,
                                            [BOTTOM] = row,    [TOP]   = row };
    MPI_Aint     sdispls[NUM_NEIGHBORS] = { [LEFT]   = DISP(1, 1),                  // first own column
                                            [RIGHT]  = DISP(imaxLocal, 1),          // last own column
                                            [BOTTOM] = DISP(1, 1),                  // first own row
                                            [TOP]    = DISP(1, jmaxLocal) };        // last own row
    MPI_Aint     rdispls[NUM_NEIGHBORS] = { [LEFT]   = DISP(0, 1),                  // left ghost column
                                            [RIGHT]  = DISP(imaxLocal + 1, 1),      // right ghost column
                                            [BOTTOM] = DISP(1, 0),                  // bottom ghost row
                                            [TOP]    = DISP(1, jmaxLocal + 1) };    // top ghost row

MPI_Neighbor_alltoallw(grid, counts, sdispls, types, grid, counts, rdispls, types, comm);

print_grid(comm, grid, imaxLocal, jmaxLocal, 2, 0);

MPI_Type_free(&column);
MPI_Type_free(&row);
MPI_Comm_free(&comm);
free(grid);

In an iterative solver, the same exchange is repeated in every iteration. Two variants help here:

* `MPI_Ineighbor_alltoallw()` is the nonblocking version: start the exchange, update the inner cells that don't depend on ghost cells, then wait and update the cells next to the ghost layers.
* `MPI_Neighbor_alltoallw_init()` (MPI 4.0) creates a **persistent** request once; every iteration then only calls `MPI_Start()` and `MPI_Wait()`, and the library can prepare the communication in advance.

## General Graph Topologies

Not every communication pattern is a grid.
MPI also supports **graph topologies**, in which every process has an arbitrary list of neighbors: `MPI_Dist_graph_create_adjacent()` (preferred, scalable) and `MPI_Dist_graph_create()`, and the older `MPI_Graph_create()`.
The neighborhood collectives work on them as well, with the neighbors in the order of the adjacency lists.
Query functions such as `MPI_Topo_test()` and `MPI_Dist_graph_neighbors()` return the type of topology and the neighbors.
They are the topic of the notebook Graph Topologies and Sparse Matrix-Vector Multiplication.

### Exercise 1: Case Study — A Parallel 2D Laplace Solver

Solve the Laplace equation $\Delta u = 0$ on the unit square with the boundary condition $u = x + y$ — whose exact solution is $u = x + y$ everywhere — with the **Jacobi method** on a grid of `imax × jmax` inner points:

$$u^{\text{new}}_{i,j} = \tfrac14\left(u_{i-1,j} + u_{i+1,j} + u_{i,j-1} + u_{i,j+1}\right)$$

The cell below already contains the process grid, the local tiles with the boundary condition in the ghost cells at the global boundary (they are never overwritten, since the neighbors there are `MPI_PROC_NULL`), the row and column types, the Jacobi update, and the comparison with the exact solution.
Complete it:

1. Exchange the ghost cells of `u` in every iteration — as in the section above.
2. Compute the largest change of all ranks, so that all ranks stop after the same iteration.

Then run it with `-n 1, 2, 4, 6, 8`.
Is the number of iterations the same for any number of processes? How does the runtime change?
The program stops when the largest change is below `tol`, but the error compared with the exact solution is much larger — why?

**Bonus:** replace the eight point-to-point calls with one `MPI_Neighbor_alltoallw()`.

<details>
<summary>Solution</summary>

```c
// 1. halo exchange
// the tags: the direction of travel of a message
enum { UPWARD, DOWNWARD, RIGHTWARD, LEFTWARD };

// the requests: one receive from and one send to every neighbor
enum { RECV_BOTTOM, RECV_TOP, RECV_LEFT, RECV_RIGHT,
       SEND_TOP, SEND_BOTTOM, SEND_RIGHT, SEND_LEFT, NUM_REQUESTS };

MPI_Request requests[NUM_REQUESTS];
MPI_Irecv(&U(1, 0),             1, row,    bottom, UPWARD,    comm, &requests[RECV_BOTTOM]);
MPI_Irecv(&U(1, jmaxLocal + 1), 1, row,    top,    DOWNWARD,  comm, &requests[RECV_TOP]);
MPI_Irecv(&U(0, 1),             1, column, left,   RIGHTWARD, comm, &requests[RECV_LEFT]);
MPI_Irecv(&U(imaxLocal + 1, 1), 1, column, right,  LEFTWARD,  comm, &requests[RECV_RIGHT]);
MPI_Isend(&U(1, jmaxLocal),     1, row,    top,    UPWARD,    comm, &requests[SEND_TOP]);
MPI_Isend(&U(1, 1),             1, row,    bottom, DOWNWARD,  comm, &requests[SEND_BOTTOM]);
MPI_Isend(&U(imaxLocal, 1),     1, column, right,  RIGHTWARD, comm, &requests[SEND_RIGHT]);
MPI_Isend(&U(1, 1),             1, column, left,   LEFTWARD,  comm, &requests[SEND_LEFT]);
MPI_Waitall(NUM_REQUESTS, requests, MPI_STATUSES_IGNORE);

// 2. the largest change of all ranks (replaces change = local_change)
MPI_Allreduce(&local_change, &change, 1, MPI_DOUBLE, MPI_MAX, comm);
```

Since the maximum is exact and the Jacobi update of a cell only depends on the previous iteration, the result — and the number of iterations — is identical for any number of processes.
With a small grid like this one, the runtime is dominated by communication: an `MPI_Allreduce()` in every iteration is expensive, and real codes check for convergence only every few iterations.

The change per iteration is not the error: Jacobi reduces the error by a factor of about $\rho \approx \cos(\pi h) \approx 0.9988$ per iteration, so the remaining error is about $\text{change} / (1 - \rho)$ — about 850 times the change.
A small change only means slow progress, not a small error.

Bonus: define `counts`, `types`, `sdispls` and `rdispls` as in the section above, once before the loop.
The displacements are relative to the buffer, so they remain valid after `u` and `unew` are swapped:

```c
MPI_Neighbor_alltoallw(u, counts, sdispls, types, u, counts, rdispls, types, comm);
```

As in the section above, passing `u` as both buffers is common practice, but strictly not standard-conforming.
</details>

In [ ]:
%%mpi -n 4 -t
const int imax = 64, jmax = 64;                 // global number of inner grid points
const double tol = 1e-6;                        // stop when the largest change is below tol
const int maxiter = 100000;

// process grid and neighbors; dimension 0 runs along i, dimension 1 along j
enum { DIM_I, DIM_J, NUM_DIMS };
int dims[NUM_DIMS] = { 0, 0 }, periods[NUM_DIMS] = { 0, 0 }, coords[NUM_DIMS];
MPI_Dims_create(size, NUM_DIMS, dims);
MPI_Comm comm;
MPI_Cart_create(MPI_COMM_WORLD, NUM_DIMS, dims, periods, 0, &comm);
MPI_Comm_rank(comm, &rank);
MPI_Cart_coords(comm, rank, NUM_DIMS, coords);
int left, right, bottom, top;
MPI_Cart_shift(comm, DIM_I, 1, &left, &right);
MPI_Cart_shift(comm, DIM_J, 1, &bottom, &top);

// size and offset of the local tile in the global grid
#define SIZE_OF(c, n, p)  ((n) / (p) + ((c) < (n) % (p) ? 1 : 0))
#define START_OF(c, n, p) ((c) * ((n) / (p)) + ((c) < (n) % (p) ? (c) : (n) % (p)))
const int imaxLocal = SIZE_OF(coords[DIM_I], imax, dims[DIM_I]), ioff = START_OF(coords[DIM_I], imax, dims[DIM_I]);
const int jmaxLocal = SIZE_OF(coords[DIM_J], jmax, dims[DIM_J]), joff = START_OF(coords[DIM_J], jmax, dims[DIM_J]);
const int ld = imaxLocal + 2;   // leading dimension: length of a row incl. ghost cells = stride from row j to row j + 1
const double h = 1.0 / (imax + 1);              // global grid points x = gi * h with gi = 0 ... imax + 1

double *u    = calloc((size_t)ld * (jmaxLocal + 2), sizeof(double));
double *unew = calloc((size_t)ld * (jmaxLocal + 2), sizeof(double));
#define U(i, j)    u[(j) * ld + (i)]
#define UNEW(i, j) unew[(j) * ld + (i)]
#define X(i) ((ioff + (i)) * h)
#define Y(j) ((joff + (j)) * h)

// boundary condition u = x + y in the ghost cells at the global boundary, in both arrays
for (int j = 0; j <= jmaxLocal + 1; ++j) {
    if (left  == MPI_PROC_NULL) U(0, j)             = UNEW(0, j)             = X(0) + Y(j);
    if (right == MPI_PROC_NULL) U(imaxLocal + 1, j) = UNEW(imaxLocal + 1, j) = X(imaxLocal + 1) + Y(j);
}
for (int i = 0; i <= imaxLocal + 1; ++i) {
    if (bottom == MPI_PROC_NULL) U(i, 0)             = UNEW(i, 0)             = X(i) + Y(0);
    if (top    == MPI_PROC_NULL) U(i, jmaxLocal + 1) = UNEW(i, jmaxLocal + 1) = X(i) + Y(jmaxLocal + 1);
}

MPI_Datatype column, row;
MPI_Type_vector(jmaxLocal, 1, ld, MPI_DOUBLE, &column);
MPI_Type_contiguous(imaxLocal, MPI_DOUBLE, &row);
MPI_Type_commit(&column);
MPI_Type_commit(&row);

int iter;
double change = 1.0;
for (iter = 0; iter < maxiter && change > tol; ++iter) {
    // TODO 1: exchange the ghost cells of u with the four neighbors

    // Jacobi update of the own cells
    double local_change = 0.0;
    for (int j = 1; j <= jmaxLocal; ++j)
        for (int i = 1; i <= imaxLocal; ++i) {
            UNEW(i, j) = 0.25 * (U(i - 1, j) + U(i + 1, j) + U(i, j - 1) + U(i, j + 1));
            double diff = UNEW(i, j) - U(i, j);
            if (diff < 0.0) diff = -diff;
            if (diff > local_change) local_change = diff;
        }
    double *tmp = u;
    u = unew;
    unew = tmp;

    // TODO 2: the largest change of all ranks
    change = local_change;
}

// compare with the exact solution u = x + y
double local_err = 0.0, err;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imaxLocal; ++i) {
        double diff = U(i, j) - (X(i) + Y(j));
        if (diff < 0.0) diff = -diff;
        if (diff > local_err) local_err = diff;
    }
MPI_Reduce(&local_err, &err, 1, MPI_DOUBLE, MPI_MAX, 0, comm);
if (rank == 0)
    printf("%d x %d processes: %d iterations, last change %.1e, max. error %.1e\n", dims[DIM_I], dims[DIM_J], iter, change, err);

MPI_Type_free(&column);
MPI_Type_free(&row);
MPI_Comm_free(&comm);
free(u);
free(unew);

## Summary

* Virtual topologies give processes coordinates and neighbors: `MPI_Dims_create()` chooses a balanced grid, `MPI_Cart_create()` creates it, and the last dimension varies fastest.
* `MPI_Cart_shift()` returns the neighbors in one dimension; `MPI_PROC_NULL` handles the boundaries without special cases.
* `MPI_Cart_sub()` splits a grid into row and column communicators.
* A 2D ghost cell exchange computes the tile sizes from the coordinates, sends rows and columns with derived types, and uses one tag per direction of travel.
* Neighborhood collectives communicate with all neighbors of a topology in a single call; `MPI_Neighbor_alltoallw()` expresses a complete ghost cell exchange.

| Call | Purpose |
|---|---|
| `MPI_Dims_create()`, `MPI_Cart_create()` | create a Cartesian process grid |
| `MPI_Cart_coords()`, `MPI_Cart_rank()`, `MPI_Cart_get()` | coordinates ↔ ranks |
| `MPI_Cart_shift()` | neighbors in one dimension |
| `MPI_Cart_sub()` | row and column communicators |
| `MPI_Neighbor_allgather(v)()`, `MPI_Neighbor_alltoall(v,w)()` | collective communication with the neighbors |
| `MPI_Ineighbor_alltoallw()`, `MPI_Neighbor_alltoallw_init()` | nonblocking and persistent variants |

[Back to the overview](../overview.ipynb)